# Analyse Agriculture through Time

Compare cropping patterns, their shares of recorded cropped area, and broader land cover through the years.

Run the cells in order. Change the place, identifier or columns to explore other records. Downloads from GeoLibre use your selected tehsil; these templates start with Hilsa, Nalanda, Bihar.


## Set up Python

Run the collapsed setup cells. They import the libraries and define `read_json`, a small response reader. It reads JSON text, treats non-standard `NaN` and `Infinity` numbers as missing, and also accepts JSON returned inside a string. HTTP errors and malformed responses remain visible. Expand the cells to read the code.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import re
import ast
import json
from getpass import getpass
from inspect import isawaitable
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


In [ ]:
SCOPE = json.loads("{\"state\": \"Bihar\", \"district\": \"Nalanda\", \"tehsil\": \"Hilsa\"}")
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


In [ ]:
"""Small response reader embedded in the notebooks' collapsed setup cell."""
import json


def read_json(response):
    """Read JSON text; represent non-standard NaN/Infinity values as missing."""
    response.raise_for_status()
    raw_text = response.text.lstrip("\ufeff")
    try:
        # Some API tables contain bare NaN or Infinity, which are not JSON numbers.
        data = json.loads(raw_text, parse_constant=lambda value: None)
        # Also accept a JSON document returned as a JSON-encoded string.
        if isinstance(data, str):
            data = json.loads(data.lstrip("\ufeff"), parse_constant=lambda value: None)
        return data
    except ValueError as error:
        raise ValueError(
            "The server response is not readable JSON. "
            "Inspect response.status_code and response.text[:500], then retry the request."
        ) from error


## Choose the place and set your API key

Edit `SCOPE` in the setup cell to change the place. The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains registration and API keys. This cell reuses `CORE_STACK_API_KEY` or asks for it privately, then stores it in this kernel’s environment. The key is sent only to the API, in the `X-API-Key` header. Restart the kernel and run from the top after changing places.


In [ ]:
place = {key: re.sub(r"[\s_]+", "_", SCOPE[key].replace("(", "").replace(")", "")).strip("_").lower()
         for key in ["state", "district", "tehsil"]}
state, district, tehsil = place["state"], place["district"], place["tehsil"]
api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key
os.environ["CORE_STACK_API_KEY"] = str(api_key).strip()
api_headers = {"X-API-Key": os.environ["CORE_STACK_API_KEY"]}
display(place)


## Read the tehsil and choose a micro-watershed

One request returns the tehsil’s tables. The cell keeps the tables used here, lists identifiers and shows the first record’s first ten fields. Change the selected identifier, then rerun the following cells. Blank fields mean the source did not supply a value.


In [ ]:
response = requests.get(API_URL + "get_tehsil_data/", params=place, headers=api_headers, timeout=180)
api_data = read_json(response)
required_tables = ['croppingIntensity_annual', 'lulc_vector']
tables = {name: pd.DataFrame(api_data.get(name, [])) for name in required_tables}
display(pd.DataFrame({"Table": required_tables, "Rows": [len(tables[name]) for name in required_tables]}))
mws_table = tables['croppingIntensity_annual']
display(mws_table[["uid"]])
mws_id = str(mws_table.iloc[0]["uid"])  # Choose another ID from the list.
selected = mws_table.loc[mws_table["uid"].astype(str) == mws_id].iloc[0]
display(selected.iloc[:10].to_frame("First 10 fields"))


## Discover data and descriptions in STAC

STAC lists published datasets, field descriptions, downloads and styles. Change `dataset` to another item from the collection. Asset links are used as published, wherever the files are hosted. STAC describes asset fields; API tables may use different names and units, which are shown explicitly in the examples below.


In [ ]:
collection_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/collection.json")
response = requests.get(collection_url, timeout=90)
collection = read_json(response)
items = pd.DataFrame([{"Item": link["href"].split("/")[-1].removesuffix(".json"),
                       "URL": urljoin(collection_url, link["href"])}
                      for link in collection["links"] if link["rel"] == "item"], columns=["Item", "URL"])
# Follow a relevant item link from the collection.
dataset = "cropping_intensity_vector"
matches = items.loc[items["Item"].str.endswith("_" + dataset)]
item = None
if not matches.empty:
    item_url = matches.iloc[0]["URL"]
    response = requests.get(item_url, timeout=90)
    item = read_json(response)
    display(Markdown(item["properties"].get("description", "No description published.")))
    field_notes = pd.DataFrame(item["properties"].get("table:columns", []))
    display(field_notes.reindex(columns=["name", "type", "description"]).head(12))
    print("Published field count:", len(field_notes), "— use field_notes to see them all.")
    display(pd.DataFrame(item["assets"]).T.reindex(columns=["title", "type", "href"]))
else:
    print("This dataset is not listed in the tehsil's STAC collection. Available items:")
    display(items)


## Cropping area and composition

API area fields are already hectares. The four categories describe land cropped once, twice or three times; do not multiply area by the crop count. Shares use the sum of these categories, not total MWS area. These categories do not establish fallow area.


In [ ]:
crop_fields = {"single_kharif_cropped_area_in_ha": "Single Kharif", "single_non_kharif_cropped_area_in_ha": "Single non-Kharif",
               "doubly_cropped_area_in_ha": "Double cropped", "triply_cropped_area_in_ha": "Triple cropped"}
cropping = pd.DataFrame({label: [selected.get(f"{field}_{y}-{y+1}") for y in YEARS] for field, label in crop_fields.items()}, index=YEARS).apply(pd.to_numeric, errors="coerce")
display(cropping.rename_axis("Starting year"))
# Only complete years enter stacked charts; missing categories are not zero.
complete = cropping.dropna()
shares = complete.div(complete.sum(axis=1).replace(0, float("nan")), axis=0) * 100
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
if not complete.empty:
    complete.plot.bar(stacked=True, ax=axes[0], ylabel="Area (ha)", rot=0, title=f"Cropping patterns · {mws_id}")
    shares.plot.bar(stacked=True, ax=axes[1], ylabel="Recorded cropped area (%)", rot=0, legend=False)
    axes[1].set_ylim(0, 100)
plt.tight_layout()
plt.show()
print("Years omitted from stacked charts:", cropping.index[cropping.isna().any(axis=1)].tolist())
fallow_fields = [name for name in selected.index if "fallow" in name or "uncropped" in name]
display(selected.reindex(fallow_fields).to_frame("Recorded fallow / uncropped values"))


## Broader land cover

Compare the main land-cover area fields for the same years. LULC columns use the starting year alone. Water-season categories are excluded here because they overlap in time.


In [ ]:
land = tables["lulc_vector"].set_index("uid").reindex([mws_id]).iloc[0]
land_fields = {"cropland": "Cropland", "tree_forest": "Trees / forest", "shrub_scrub": "Shrub / scrub", "barrenlands": "Barren land", "built-up": "Built-up"}
land_cover = pd.DataFrame({label: [land.get(f"{field}_area_in_ha_{y}") for y in YEARS] for field, label in land_fields.items()}, index=YEARS).apply(pd.to_numeric, errors="coerce")
display(land_cover)
land_cover.plot(subplots=True, layout=(2, 3), figsize=(12, 6), marker="o", legend=False, ylabel="Area (ha)", sharey=False)
plt.suptitle(f"Land-cover area · {mws_id}")
plt.tight_layout()
plt.show()


## Cropping intensity

Cropping intensity is a unitless value reported by the API. Compare its variation with the area charts above.


In [ ]:
intensity = pd.Series([selected.get(f"cropping_intensity_unit_less_{y}-{y+1}") for y in YEARS], index=YEARS, dtype=float)
intensity.plot(figsize=(8, 3), marker="o", ylabel="Cropping intensity (unitless)", title=f"Cropping intensity · {mws_id}")
plt.tight_layout()
plt.show()
